# Retail Silver ETL — AWS Glue / PySpark

Notebook converted from the exported AWS Glue job `retail-silver-etl`.

## Exported Glue job configuration
- **Glue version:** 6.0
- **Worker type:** G.1X
- **Workers:** 10
- **Timeout:** 480 minutes
- **Job bookmark:** job-bookmark-disable

> The code cells preserve the script embedded in the Glue export. Execution requires a compatible Spark/AWS Glue environment and access to the referenced S3 paths.


## ETL code


In [ ]:
import sys
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from awsglue.context import GlueContext
from awsglue.job import Job

# ============================================================


## INITIALIZE GLUE


In [ ]:
# 1. INITIALIZE GLUE
# ============================================================

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

job = Job(glueContext)
job.init("retail-silver-etl", {})

BUCKET = "tus-retail-clothing-store-datasets"

BRONZE = f"s3://{BUCKET}/bronze/"
SILVER = f"s3://{BUCKET}/silver/"
REJECTED = f"s3://{BUCKET}/data_quality/rejected_records/"

# ============================================================


## COMMON HELPER FUNCTIONS


In [ ]:
# 2. COMMON HELPER FUNCTIONS
# ============================================================

def read_bronze(dataset):
    path = BRONZE + dataset + "/"
    df = spark.read.parquet(path)
    print(f"{dataset}: Bronze rows = {df.count()}")
    return df


def write_silver(df, dataset):
    path = SILVER + dataset + "/"
    df.write.mode("overwrite").parquet(path)
    print(f"{dataset}: Silver rows = {df.count()}")
    print(f"Silver output: {path}")


def write_rejected(df, dataset, reason):
    """
    Save removed records with their original columns
    and a reason for rejection.
    """
    rejected_df = (
        df.withColumn("rejection_reason", F.lit(reason))
          .withColumn("source_dataset", F.lit(dataset))
          .withColumn("rejected_at", F.current_timestamp())
    )

    if rejected_df.limit(1).count() > 0:
        path = REJECTED + dataset + "/"
        rejected_df.write.mode("overwrite").parquet(path)
        print(f"{dataset}: Rejected rows saved to {path}")
    else:
        print(f"{dataset}: No rejected rows for {reason}")


def split_duplicates(df, key_columns):
    """
    Keep one record per key and return duplicate records separately.
    """
    window = Window.partitionBy(*key_columns).orderBy(
        F.col("source_file").asc_nulls_last(),
        F.col("ingestion_timestamp").asc_nulls_last()
    )

    ranked = df.withColumn("_duplicate_rank", F.row_number().over(window))

    kept = ranked.filter(F.col("_duplicate_rank") == 1).drop("_duplicate_rank")
    rejected = (
        ranked.filter(F.col("_duplicate_rank") > 1)
              .drop("_duplicate_rank")
    )

    return kept, rejected


# ============================================================


## BRANDS


In [ ]:
# 3. BRANDS
# ============================================================

print("\n========== BRANDS ==========")

df_brands = read_bronze("brands")

# Trim whitespace from brand_name
df_brands = df_brands.withColumn(
    "brand_name",
    F.trim(F.col("brand_name"))
)

# Remove special characters from brand_code
df_brands = df_brands.withColumn(
    "brand_code",
    F.regexp_replace(F.col("brand_code"), r"[^A-Za-z0-9]", "")
)

# Standardize category codes
brand_anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS": "TOY"
}

df_brands = df_brands.replace(
    to_replace=brand_anomalies,
    subset=["category_code"]
)

write_silver(df_brands, "brands")


# ============================================================


## CATEGORY


In [ ]:
# 4. CATEGORY
# ============================================================

print("\n========== CATEGORY ==========")

df_category = read_bronze("category")

# Remove duplicate category_code records.
# Preserve removed rows with a rejection reason.
df_category, rejected_category = split_duplicates(
    df_category,
    ["category_code"]
)

write_rejected(
    rejected_category,
    "category",
    "DUPLICATE_CATEGORY_CODE"
)

# Convert category_code to uppercase
df_category = df_category.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
)

write_silver(df_category, "category")


# ============================================================


## PRODUCTS


In [ ]:
# 5. PRODUCTS
# ============================================================

print("\n========== PRODUCTS ==========")

df_products = read_bronze("products")

# Remove 'g' from weight_grams and cast to integer
df_products = df_products.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast("int")
)

# Replace comma with decimal point in length_cm
df_products = df_products.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".").cast("float")
)

# Convert category_code and brand_code to uppercase
df_products = (
    df_products
    .withColumn("category_code", F.upper(F.col("category_code")))
    .withColumn("brand_code", F.upper(F.col("brand_code")))
)

# Correct spelling mistakes in material
df_products = df_products.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)

# Convert negative rating_count to positive.
# Replace null rating_count with 0.
df_products = df_products.withColumn(
    "rating_count",
    F.when(
        F.col("rating_count").isNotNull(),
        F.abs(F.col("rating_count").cast("int"))
    ).otherwise(F.lit(0))
)

write_silver(df_products, "products")


# ============================================================


## CUSTOMERS


In [ ]:
# 6. CUSTOMERS
# ============================================================

print("\n========== CUSTOMERS ==========")

df_customers = read_bronze("customers")

# Remove rows where customer_id is NULL.
# Preserve those rows in rejected_records.
rejected_customers = df_customers.filter(
    F.col("customer_id").isNull()
)

write_rejected(
    rejected_customers,
    "customers",
    "NULL_CUSTOMER_ID"
)

df_customers = df_customers.filter(
    F.col("customer_id").isNotNull()
)

# Replace NULL phone values with 'Not Available'
df_customers = df_customers.fillna(
    "Not Available",
    subset=["phone"]
)

write_silver(df_customers, "customers")


# ============================================================


## DATE / CALENDAR


In [ ]:
# 7. DATE / CALENDAR
# ============================================================

print("\n========== DATE ==========")

df_date = read_bronze("date")

# Convert date from string to DateType
df_date = df_date.withColumn(
    "date",
    F.to_date(F.col("date"), "dd-MM-yyyy")
)

# Remove duplicate dates and preserve removed rows
df_date, rejected_date = split_duplicates(
    df_date,
    ["date"]
)

write_rejected(
    rejected_date,
    "date",
    "DUPLICATE_DATE"
)

# Capitalize first letter of day_name
df_date = df_date.withColumn(
    "day_name",
    F.initcap(F.col("day_name"))
)

# Convert negative week_of_year values to positive
df_date = df_date.withColumn(
    "week_of_year",
    F.abs(F.col("week_of_year").cast("int"))
)

# Format quarter as Q3-2025, etc.
df_date = df_date.withColumn(
    "quarter",
    F.concat(
        F.lit("Q"),
        F.col("quarter").cast("string"),
        F.lit("-"),
        F.col("year").cast("string")
    )
)

# Format week as Week31-2025, etc.
df_date = df_date.withColumn(
    "week_of_year",
    F.concat(
        F.lit("Week"),
        F.col("week_of_year").cast("string"),
        F.lit("-"),
        F.col("year").cast("string")
    )
)

# Rename week_of_year to week
df_date = df_date.withColumnRenamed(
    "week_of_year",
    "week"
)

write_silver(df_date, "date")


# ============================================================


## ORDER ITEMS


In [ ]:
# 8. ORDER ITEMS
# ============================================================

print("\n========== ORDER ITEMS ==========")

df_orders = read_bronze("order_items")

# Remove duplicate order_id + item_seq records
df_orders, rejected_orders = split_duplicates(
    df_orders,
    ["order_id", "item_seq"]
)

write_rejected(
    rejected_orders,
    "order_items",
    "DUPLICATE_ORDER_ID_ITEM_SEQ"
)

# Convert quantity: 'Two' -> 2, then integer
df_orders = df_orders.withColumn(
    "quantity",
    F.when(F.col("quantity") == "Two", 2)
     .otherwise(F.col("quantity"))
     .cast("int")
)

# Remove '$' from unit_price and cast to double
df_orders = df_orders.withColumn(
    "unit_price",
    F.regexp_replace(F.col("unit_price"), r"[$]", "").cast("double")
)

# Remove '%' from discount_pct and cast to double
df_orders = df_orders.withColumn(
    "discount_pct",
    F.regexp_replace(F.col("discount_pct"), "%", "").cast("double")
)

# Convert coupon_code to lowercase after trimming
df_orders = df_orders.withColumn(
    "coupon_code",
    F.lower(F.trim(F.col("coupon_code")))
)

# Standardize channel
df_orders = df_orders.withColumn(
    "channel",
    F.when(F.col("channel") == "web", "Website")
     .when(F.col("channel") == "app", "Mobile")
     .otherwise(F.col("channel"))
)

# Convert dt from string to date
df_orders = df_orders.withColumn(
    "dt",
    F.to_date(F.col("dt"), "yyyy-MM-dd")
)

# Convert order_ts to timestamp.
# Use the second format as fallback.
df_orders = df_orders.withColumn(
    "order_ts",
    F.coalesce(
        F.to_timestamp(
            F.col("order_ts"),
            "yyyy-MM-dd HH:mm:ss"
        ),
        F.to_timestamp(
            F.col("order_ts"),
            "dd-MM-yyyy HH:mm"
        )
    )
)

# Convert item_seq to integer
df_orders = df_orders.withColumn(
    "item_seq",
    F.col("item_seq").cast("int")
)

# Remove non-numeric symbols from tax_amount and cast to double
df_orders = df_orders.withColumn(
    "tax_amount",
    F.regexp_replace(
        F.col("tax_amount"),
        r"[^0-9.\-]",
        ""
    ).cast("double")
)

# Add processing timestamp
df_orders = df_orders.withColumn(
    "processed_time",
    F.current_timestamp()
)

write_silver(df_orders, "order_items")


# ============================================================


## COMPLETE JOB


In [ ]:
# 9. COMPLETE JOB
# ============================================================

print("\n========== SILVER ETL COMPLETED ==========")

job.commit()
